# Notebook 14: Generative Augmentation (Text Generation)

## Objetivo
Generar pares sinteticos para el nivel hard usando Ollama y un cambio fuerte de estilo.

Este notebook es breve y funciona como extra exploratorio para data augmentation.


## 1. Imports


In [10]:
import json
import random
import pandas as pd
import time
from pathlib import Path
import requests
from typing import Dict, List, Optional

## 2. Configuracion

Requiere un servidor local de Ollama en `http://localhost:11434`.
Ejemplo: `ollama pull qwen2.5:7b-instruct`


In [3]:
# --- Configuración de Rutas ---
PROJECT_DIR = Path('..').resolve()
# Ajusta esta ruta si tu estructura de carpetas varía
DATA_DIR = PROJECT_DIR / 'data' / 'raw' / 'hard' / 'train'
OUTPUT_DIR = PROJECT_DIR / 'outputs' / 'synthetic'
OUTPUT_PATH = OUTPUT_DIR / 'synthetic_data.jsonl'

# --- Configuración de Ollama ---
# Usamos el endpoint de chat, es más robusto para modelos 'instruct'
OLLAMA_BASE_URL = 'http://localhost:11434'
OLLAMA_CHAT_URL = f'{OLLAMA_BASE_URL}/api/chat'
MODEL_NAME = 'qwen3:14b'

# --- Parámetros de Generación ---
N_SAMPLES = 12       # Número de muestras a generar
MIN_LEN = 40         # Longitud mínima de texto para considerarlo
TIMEOUT_SEC = 120    # Tiempo de espera máximo para la inferencia
random.seed(42)

# --- Validaciones Previas ---
if not DATA_DIR.exists():
    raise FileNotFoundError(f'Directorio de datos no encontrado: {DATA_DIR}')

# 1. Verificar si Ollama está corriendo
try:
    requests.get(OLLAMA_BASE_URL)
    print(f"Ollama detectado en {OLLAMA_BASE_URL}")
except requests.exceptions.ConnectionError:
    raise ConnectionError(
        "No se puede conectar a Ollama. "
        "Asegúrate de ejecutar 'ollama serve' en una terminal aparte."
    )

print(f'Model target: {MODEL_NAME}')
print(f'Data dir: {DATA_DIR}')

Ollama detectado en http://localhost:11434
Model target: qwen3:14b
Data dir: /home/eeguskiza/DEUSTO/multi-author-analysis/data/raw/hard/train


## 3. Cargar muestra del dataset hard


In [4]:
def load_hard_sentences(data_dir: Path, min_len: int = 40, max_sentences: int = 200) -> List[str]:
    """
    Carga oraciones del dataset 'hard' filtrando por longitud.
    """
    files = sorted(data_dir.glob('problem-*.txt'))
    random.shuffle(files)

    sentences = []
    
    # Iteramos sobre los ficheros hasta llenar el cupo
    for path in files:
        try:
            with path.open(encoding='utf-8', errors='ignore') as f:
                for line in f:
                    text = line.strip()
                    # Filtros básicos de calidad
                    if not text or len(text) < min_len:
                        continue
                    
                    sentences.append(text)
                    if len(sentences) >= max_sentences:
                        return sentences
        except Exception as e:
            print(f"⚠️ Error leyendo {path.name}: {e}")
            continue
            
    return sentences

# Ejecución de carga
all_sentences = load_hard_sentences(DATA_DIR, min_len=MIN_LEN)

if not all_sentences:
    raise ValueError('No se encontraron oraciones válidas en el directorio.')

# Seleccionamos la muestra aleatoria para el experimento
sample_texts = random.sample(all_sentences, k=min(N_SAMPLES, len(all_sentences)))

print(f'Cargadas {len(all_sentences)} oraciones candidatas.')
print(f'Ejemplo de muestra: "{sample_texts[0][:100]}..."')

Cargadas 200 oraciones candidatas.
Ejemplo de muestra: "an explanation what it is really about and maybe workthrough by beginning with the priorities...."


## 4. Prompt generativo (Text Generation) y llamada a Ollama


In [5]:
def generate_style_change(text_a: str, model: str = MODEL_NAME, url: str = OLLAMA_CHAT_URL) -> str:
    """
    Envía el texto a Ollama para reescribirlo con un cambio de estilo drástico.
    Usa el endpoint de chat para mejor adherencia a instrucciones.
    """
    
    # Prompt del Sistema: Define el comportamiento esperado
    system_prompt = (
        "You are a data augmentation expert. Your task is to rewrite the input text "
        "changing its style completely (e.g., formal to slang, complex to simple, abstract to concrete). "
        "Maintain the original meaning but change the vocabulary and tone drastically. "
        "OUTPUT ONLY THE NEW SENTENCE. NO PREAMBLE."
    )

    # Payload para la API de Chat
    payload = {
        'model': model,
        'messages': [
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': f"Input Text: {text_a}"}
        ],
        'stream': False,
        'options': {
            'temperature': 0.8  # Un poco de creatividad para el cambio de estilo
        }
    }

    try:
        response = requests.post(url, json=payload, timeout=TIMEOUT_SEC)
        
        # Si el modelo no existe, Ollama suele devolver 404 o 400 con mensaje específico
        if response.status_code == 404:
            raise ValueError(f"Modelo '{model}' no encontrado o endpoint incorrecto. ¿Hiciste 'ollama pull {model}'?")
            
        response.raise_for_status()
        data = response.json()
        
        # Extraer contenido del mensaje
        output = data.get('message', {}).get('content', '').strip()
        
        if not output:
            raise ValueError(f'Respuesta vacía del modelo.')
            
        return output

    except requests.exceptions.RequestException as e:
        print(f"Error de red/API: {e}")
        return "" # Devolvemos vacío para manejarlo en el loop

## 5. Probar con un ejemplo real


In [7]:
# Test rápido con el primer elemento
sample_text = sample_texts[0]

print("--- Test de Generación ---")
print(f"ORIGINAL: {sample_text}")

start_t = time.time()
new_text = generate_style_change(sample_text)
elapsed = time.time() - start_t

if new_text:
    print(f"\nGENERADO ({elapsed:.2f}s): {new_text}")
else:
    print("\nFalló la generación en el test.")

--- Test de Generación ---
ORIGINAL: an explanation what it is really about and maybe workthrough by beginning with the priorities.

GENERADO (76.39s): A thorough breakdown of its core essence, paired with a step-by-step walkthrough that starts by tackling the most crucial elements first.


## 6. Generar pares sinteticos


In [8]:
synthetic_data = []

print(f"--- Iniciando generación de {len(sample_texts)} pares ---")

for i, text_a in enumerate(sample_texts):
    print(f"[{i+1}/{len(sample_texts)}] Procesando...", end='\r')
    
    text_b = generate_style_change(text_a)
    
    # Solo guardamos si la generación fue exitosa
    if text_b:
        synthetic_data.append({
            'text_a': text_a,
            'text_b': text_b,
            'label': 1, # 1 indica que es un par del "mismo autor" (semánticamente) pero distinto estilo
            'source': 'synthetic_style_change',
            'model': MODEL_NAME
        })
    
    # Pequeña pausa para no saturar si corres en local (opcional con GPU potente)
    # time.sleep(0.1) 

print(f"\nFinalizado. Pares generados exitosamente: {len(synthetic_data)}")
if len(synthetic_data) > 0:
    print(f"Ejemplo 1:\n A: {synthetic_data[0]['text_a'][:50]}...\n B: {synthetic_data[0]['text_b'][:50]}...")

--- Iniciando generación de 12 pares ---
[12/12] Procesando...
Finalizado. Pares generados exitosamente: 12
Ejemplo 1:
 A: an explanation what it is really about and maybe w...
 B: A breakdown of the real deal, maybe a walkthrough ...


## 7. Guardar synthetic_data.jsonl


In [9]:
# Asegurar que el directorio existe
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if synthetic_data:
    with OUTPUT_PATH.open('w', encoding='utf-8') as f:
        for row in synthetic_data:
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

    print(f'Archivo guardado en:\n{OUTPUT_PATH}')
else:
    print("No hay datos para guardar.")

Archivo guardado en:
/home/eeguskiza/DEUSTO/multi-author-analysis/outputs/synthetic/synthetic_data.jsonl


# 8. Visualizacion

In [11]:
# Configuración para ver el texto completo en las celdas
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 20)

# Cargar el fichero recién guardado
try:
    df_synthetic = pd.read_json(OUTPUT_PATH, lines=True)
    
    print(f"Resumen de datos generados en: {OUTPUT_PATH}")
    print(f"Total de pares: {len(df_synthetic)}")
    print(f"Modelo utilizado: {df_synthetic['model'].iloc[0] if not df_synthetic.empty else 'N/A'}")
    
    print("\n--- Muestra de Pares (Original vs Sintético) ---")
    # Mostramos las columnas relevantes comparando A y B
    display(df_synthetic[['text_a', 'text_b']].head(10))

except ValueError as e:
    print(f"Error leyendo el archivo JSONL (¿está vacío?): {e}")
except FileNotFoundError:
    print(f"No se encuentra el archivo: {OUTPUT_PATH}")

Resumen de datos generados en: /home/eeguskiza/DEUSTO/multi-author-analysis/outputs/synthetic/synthetic_data.jsonl
Total de pares: 12
Modelo utilizado: qwen3:14b

--- Muestra de Pares (Original vs Sintético) ---


,text_a,text_b
0,an explanation what it is really about and maybe workthrough by beginning with the priorities.,"A breakdown of the real deal, maybe a walkthrough starting with the most important stuff."
1,"Yes, the Taliban did not attack coalition forces, but , in violation of Trump's deal; meanwhile US troop withdrawals continued.","Look, the Taliban didn't go after the coalition troops, but they totally ignored that Trump deal, and at the same time, the US kept pulling out troops."
2,"Republicans are setting the stage for max pain and I can guarantee you when the debt ceiling does come up, corporate Democrats will harken back to this moment as justification for why they had to give into the Republican demands.","Republicans are messin' things up for a big ol' headache, and I swear when the debt limit comes around, those corporate Dems will look back on this moment and say, ""Yeah, that's why we had to cave in to the GOP's crap."""
3,And even if they win will the people of ukraine ever really accept their new citizenship or will there always be resentment and division.,"So even if they pull off a win, will the Ukrainian folks ever really go along with this new citizenship stuff, or will there always be some lingering bitterness and split loyalties?"
4,insulation foam in the gaps to eliminate draft as much as possible.,"Pack the gaps with foam to block drafts completely, if possible."
5,Student loan borrowers also pay taxes—if they have jobs that earns less than $billions—so it is not really the same comparison.,"Student loan borrowers still fork over taxes when their gig brings in under a few million, so comparing them to folks with way bigger paychecks isn't apples to apples."
6,Seriously they are the big winners in the Ukranian war.,"No doubt about it, they're the total victors in the Ukraine conflict, and everyone knows it."
7,"Clinton was a bad candidate who ran a bad campaign wrought with hubris, and she embodied what many people were sick of at that point in time - a status quo insider politician who secured the nomination largely in part because of her position within the party.","Clinton was a total disaster who ran a campaign that was a total mess, full of her own ego, and she was the perfect example of what people were tired of back then—a career insider who didn’t get the job done and got the nomination mostly because of her party connections."
8,"Can you explain to me how this is a neutral description of a group the judge knows nothing about, “Responsible, careful citizens with a great deal of respect and care for their firearms.”.","How’s this a balanced take on a crowd the judge’s clueless about, “Responsible, cautious folks who treat their guns like sacred relics”?"
9,"They're also the same guys who will claim an AR is the best home defense choice cuz accuracy and ballistics, but for some reason they need 30 rounds.","Same crew who'll swear ARs are the ultimate pick for home security because of precision and bullet physics, yet somehow still need 30 rounds for some bizarre reason."
